In [ ]:
# Imports & Path
# Run once at the start of every session


import sys
import os
import time
import random
import datetime

# Add Slurrybot folder to path so Python finds all files
sys.path.append(r"C:\Users\digic\OneDrive\Desktop\SlurryBotPc\Slurry Formation 2024\SlurryBot")

# Load all robot functions from the wrapper
%reload_ext autoreload
%autoreload 2
from Robot_wrapper import *

In [ ]:
# Connect Robot
# Run once at the start of every session


# Connect to robot arm
robot = Robot()
robot.initialize()
print("✓ Robot connected")

In [ ]:
robot.initialize()
robot.GoTo_InitialPoint()
robot.restart()

In [ ]:
# Connect Scale
# Run once at the start of every session


# Connect to scale
from Drivers.scale_driver import scale
scale = Scale('COM4', 9600, 10)
scale.connect()
print("✓ Scale connected")

# scale object methods:
# scale.tare()            → zero the scale
# scale.measure()         → single measurement
# scale.measure_stable()  → measure and wait until value is stable

In [ ]:
# Test Scale
lines = scale.get(CMD_PRINT)
print("Raw response from scale:", lines)

## test scales before use, True = ready
reading = scale.measure()
print(reading.value, reading.stable)

In [ ]:
# Helper Functions
# Run once – defines all helper functions used later


# ── Safety limits ───────────────────────────────────────────
# If these are hit the container is probably empty
MAX_LARGE_SCOOPS  = 20
MAX_MEDIUM_SCOOPS = 30

# ── Logging ─────────────────────────────────────────────────
log = []

def log_event(event_type, message, weight=None, added=None):
    """
    Logs any event during a weighing run.
    event_type: "scoop", "info", "warning", "error", "result"
    """
    entry = {
        "time":    datetime.datetime.now().strftime("%H:%M:%S"),
        "type":    event_type,
        "message": message,
        "weight":  weight,
        "added":   added,
    }
    log.append(entry)

    weight_str = f" | total: {weight:.4f}g" if weight is not None else ""
    added_str  = f" | added: +{added:.4f}g" if added  is not None else ""
    print(f"[{entry['time']}] [{event_type.upper():7s}] "
          f"{message}{added_str}{weight_str}")

def print_run_summary():
    """Prints full summary of the current run."""
    print("\n" + "="*60)
    print("RUN SUMMARY")
    print("="*60)
    print(f"{'Time':<10} {'Type':<10} {'Event':<25} {'Added':>8} {'Total':>8}")
    print("-"*60)
    for entry in log:
        added_str  = f"{entry['added']:>8.4f}g"  if entry['added']  is not None else f"{'':>9}"
        weight_str = f"{entry['weight']:>8.4f}g" if entry['weight'] is not None else f"{'':>9}"
        print(f"{entry['time']:<10} {entry['type'].upper():<10} "
              f"{entry['message']:<25} {added_str} {weight_str}")
    print("="*60)
    scoops = [e for e in log if e["type"] == "scoop"]
    if scoops:
        total = sum(e["added"] for e in scoops if e["added"] is not None)
        print(f"Total scoops:    {len(scoops)}")
        print(f"Total dispensed: {total:.4f}g")
    print("="*60)

# ── Weight measurement ───────────────────────────────────────

def calc_weight_needed(weight_dispensed, target_weight):
    """Returns how much weight is still needed."""
    return target_weight - weight_dispensed

def wait_for_stable_weight(previous_weight, max_attempts=30):
    """
    Waits until scale returns a stable reading.
    Raises TimeoutError if scale does not stabilise.
    """
    for _ in range(max_attempts):
        raw = scale.measure()
        if raw.stable and abs(raw.value - previous_weight) > 0.01:
            return raw.value
        time.sleep(0.1)
    raise TimeoutError("Scale did not stabilise – check scale connection!")

In [ ]:
# Position Loading & Execution - Replaces all hardcoded coordinates!
# Run once – defines load and execute functions


def load_positions(filename):
    """
    Loads all points from a jogger txt file.
    Returns ordered dict: {name: {cartesian, angular, gripper, type}}
    """
    positions = {}

    with open(filename, "r") as f:
        lines = [l for l in f.readlines() if l.strip() != ""]

    i = 0
    while i < len(lines):
        try:
            name_part, cart_vals = lines[i].split(" cartesian: ", 1)
            point_name = name_part.strip()
            cartesian  = eval(cart_vals.strip())

            _, ang_vals = lines[i+1].split(" angular: ", 1)
            angular     = eval(ang_vals.strip())

            _, grip_val = lines[i+2].split(": ", 1)
            gripper     = int(grip_val.strip())

            _, type_val = lines[i+3].split(": ", 1)
            point_type  = type_val.strip()

            positions[point_name] = {
                "cartesian": cartesian,
                "angular":   angular,
                "gripper":   gripper,
                "type":      point_type
            }
            i += 4

        except Exception as e:
            print(f"Warning: could not parse line {i}: {e}")
            i += 1

    print(f"✓ Loaded {len(positions)} points from {filename}")
    return positions


def execute_movement(positions, speed=50):
    """
    Executes all points in a positions dict in order.
    Replaces all hardcoded scoop/pickup/replace functions!
    """
    prev_gripper = None

    for point_name, point in positions.items():

        # Only update gripper if value changed
        if point["gripper"] != prev_gripper:
            robot.arm.set_gripper_position(point["gripper"], wait=True)
            prev_gripper = point["gripper"]

        if point["type"] == "linear":
            robot.arm.set_position(
                *point["cartesian"],
                speed=speed, wait=True
            )
        elif point["type"] == "angular":
            robot.arm.set_servo_angle(
                angle=point["angular"],
                speed=speed, wait=True
            )

In [ ]:
# Load All Position Files
# Run once at start of session
# Re-run individual line if you re-taught a position

'''
# Large spatula
pos_pickup_large        = load_positions("Positions/pickup_large.txt")
pos_replace_large       = load_positions("Positions/replace_large.txt")
pos_scoop_big_middle    = load_positions("Positions/scoop_big_middle.txt")
pos_scoop_big_left      = load_positions("Positions/scoop_big_left.txt")
pos_scoop_big_right     = load_positions("Positions/scoop_big_right.txt")

# Medium spatula
pos_pickup_medium       = load_positions("Positions/pickup_medium.txt")
pos_replace_medium      = load_positions("Positions/replace_medium.txt")
pos_scoop_medium_left   = load_positions("Positions/scoop_medium_left.txt")
pos_scoop_medium_right  = load_positions("Positions/scoop_medium_right.txt")
pos_scoop_medium_middle = load_positions("Positions/scoop_medium_middle.txt")
pos_scoop_medium_farleft= load_positions("Positions/scoop_medium_farleft.txt")

# Tiny scoops
pos_scoop_tiny          = load_positions("Positions/scoop_tiny.txt")
pos_scoop_tiny_left     = load_positions("Positions/scoop_tiny_left.txt")
pos_scoop_tiny_right    = load_positions("Positions/scoop_tiny_right.txt")

# Funnel
pos_place_funnel        = load_positions("Positions/place_funnel.txt")
pos_replace_funnel      = load_positions("Positions/replace_funnel.txt")

print("\n✓ All positions loaded successfully")
'''

In [ ]:
# Movement Functions
# Thin wrappers around execute_movement
# These replace ALL the old hardcoded functions!
# Speed can be adjusted per function here

'''

# Large spatula
def pickup_large():        execute_movement(pos_pickup_large,         speed=60)
def replace_big_spatula(): execute_movement(pos_replace_large,        speed=60)
def scoop_big_middle():    execute_movement(pos_scoop_big_middle,     speed=50)
def scoop_big_left():      execute_movement(pos_scoop_big_left,       speed=50)
def scoop_big_right():     execute_movement(pos_scoop_big_right,      speed=50)

# Medium spatula
def pickup_medium():          execute_movement(pos_pickup_medium,        speed=60)
def replace_medium_spatula(): execute_movement(pos_replace_medium,       speed=60)
def scoop_medium_left():      execute_movement(pos_scoop_medium_left,    speed=40)
def scoop_medium_right():     execute_movement(pos_scoop_medium_right,   speed=40)
def scoop_medium_middle():    execute_movement(pos_scoop_medium_middle,  speed=40)
def scoop_medium_farleft():   execute_movement(pos_scoop_medium_farleft, speed=40)

# Tiny scoops
def scoop_tiny():       execute_movement(pos_scoop_tiny,       speed=30)
def scoop_tiny_left():  execute_movement(pos_scoop_tiny_left,  speed=30)
def scoop_tiny_right(): execute_movement(pos_scoop_tiny_right, speed=30)

# Funnel
def place_funnel():   execute_movement(pos_place_funnel,   speed=50)
def replace_funnel(): execute_movement(pos_replace_funnel, speed=50)

print("✓ Movement functions defined")
'''

In [ ]:
# Calibration
# Run before first weighing session or when changing powder
# Compares your results to Aidan's reference values


def calibrate_scoop(scoop_function, name, repetitions=3):
    """
    Tests a scooping motion N times and prints average yield.
    Compare results to reference values to check if positions
    still match physical setup.
    """
    print(f"\n--- Calibrating: {name} ({repetitions}x) ---")
    scale.tare()

    results        = []
    previous_weight = 0

    for i in range(repetitions):
        scoop_function()
        weight  = wait_for_stable_weight(previous_weight)
        added   = weight - previous_weight
        results.append(added)
        print(f"  Scoop {i+1}: +{added:.4f}g (total: {weight:.4f}g)")
        previous_weight = weight

    avg = sum(results) / len(results)
    print(f"  → Average: {avg:.4f}g | "
          f"Min: {min(results):.4f}g | "
          f"Max: {max(results):.4f}g")
    return avg

# ── Reference values from Aidan's report ────────────────────
# If your values differ significantly → re-teach positions!
#
# Large spatula:   middle ~1.0g | left ~0.6g | right ~0.5g
# Medium spatula:  left ~0.3g   | middle ~0.23g | right ~0.21g
# Tiny:            ~0.05-0.08g range

# Uncomment what you want to calibrate:
# calibrate_scoop(scoop_big_middle,    "big middle",     repetitions=3)
# calibrate_scoop(scoop_big_left,      "big left",       repetitions=3)
# calibrate_scoop(scoop_big_right,     "big right",      repetitions=3)
# calibrate_scoop(scoop_medium_left,   "medium left",    repetitions=3)
# calibrate_scoop(scoop_medium_middle, "medium middle",  repetitions=3)
# calibrate_scoop(scoop_medium_right,  "medium right",   repetitions=3)
# calibrate_scoop(scoop_medium_farleft,"medium farleft", repetitions=3)
# calibrate_scoop(scoop_tiny,          "tiny middle",    repetitions=3)
# calibrate_scoop(scoop_tiny_left,     "tiny left",      repetitions=3)
# calibrate_scoop(scoop_tiny_right,    "tiny right",     repetitions=3)

In [ ]:
# Weighing Logic
# Internal function – called by run_weighing_workflow()

def run_weighing_logic(target_weight):
    """
    Contains all scooping and weighing logic.
    Called by run_weighing_workflow(), not directly.
    """
    global previous_weight, last_scoop

    previous_weight = 0
    last_scoop      = None

    # ── Select spatula ───────────────────────────────────────
    if target_weight >= 1.0:
        pickup_large()
        log_event("info", "Large spatula selected")
    else:
        pickup_medium()
        log_event("info", "Medium spatula selected")

    # ── Starting weight ──────────────────────────────────────
    weight_dispensed = scale.measure_stable().value
    weight_needed    = calc_weight_needed(weight_dispensed, target_weight)
    log_event("info",
              f"Start: {weight_dispensed}g | Needed: {weight_needed}g")

    # ============================================================
    # CASE A: target >= 1g → large spatula first, then medium
    # ============================================================
    if target_weight >= 1.0:

        large_scoop_count = 0

        # First scoop based on target weight
        if 1.0 <= target_weight < 1.5:
            scoop_big_right()
            last_scoop = scoop_big_right
            log_event("scoop", "big right (first)")
        elif 1.5 <= target_weight < 2.0:
            scoop_big_left()
            last_scoop = scoop_big_left
            log_event("scoop", "big left (first)")
        else:
            scoop_big_middle()
            last_scoop = scoop_big_middle
            log_event("scoop", "big middle (first)")
        large_scoop_count += 1

        weight_dispensed = wait_for_stable_weight(previous_weight)
        weight_needed    = calc_weight_needed(weight_dispensed, target_weight)
        fraction_left    = weight_needed / target_weight
        log_event("scoop", "first scoop measured",
                  weight=weight_dispensed,
                  added=weight_dispensed - previous_weight)
        previous_weight = weight_dispensed

        # ── Large spatula loop ───────────────────────────────
        # Keep scooping until only 37% of target is remaining
        while fraction_left > 0.37:

            if large_scoop_count >= MAX_LARGE_SCOOPS:
                log_event("warning",
                          f"Large limit reached ({MAX_LARGE_SCOOPS})"
                          f" – container probably empty!")
                print("⚠️ Stopping large loop – check container!")
                break

            if fraction_left > 0.62:
                scoop_big_middle()
                last_scoop = scoop_big_middle
                log_event("scoop", "big middle")
            elif 0.5 <= fraction_left <= 0.62:
                scoop_big_left()
                last_scoop = scoop_big_left
                log_event("scoop", "big left")
            elif 0.4 <= fraction_left < 0.5:
                # Second scoop same spot ≈ half the amount
                last_scoop()
                log_event("scoop", "repeat last scoop")
            elif 0.37 <= fraction_left < 0.4:
                scoop_big_right()
                last_scoop = scoop_big_right
                log_event("scoop", "big right")
            else:
                log_event("error", "Unexpected state!")
                break

            large_scoop_count += 1
            weight_dispensed = wait_for_stable_weight(previous_weight)
            weight_needed    = calc_weight_needed(weight_dispensed, target_weight)
            fraction_left    = weight_needed / target_weight
            log_event("scoop", f"large #{large_scoop_count}",
                      weight=weight_dispensed,
                      added=weight_dispensed - previous_weight)
            previous_weight = weight_dispensed

        # ── Switch to medium spatula ─────────────────────────
        if 0.03 <= fraction_left < 0.37:
            log_event("info", "Switching to medium spatula")
            replace_big_spatula()
            pickup_medium()

            medium_scoop_count = 0

            while weight_needed > 0.05:

                if medium_scoop_count >= MAX_MEDIUM_SCOOPS:
                    log_event("warning",
                              f"Medium limit reached ({MAX_MEDIUM_SCOOPS})"
                              f" – container probably empty!")
                    print("⚠️ Stopping medium loop – check container!")
                    break

                if weight_needed >= 0.6:
                    scoop_medium_left()
                    log_event("scoop", "medium left")
                elif 0.4 <= weight_needed < 0.6:
                    scoop_medium_middle()
                    log_event("scoop", "medium middle")
                elif 0.3 <= weight_needed < 0.4:
                    scoop_medium_right()
                    log_event("scoop", "medium right")
                elif 0.2 <= weight_needed < 0.3:
                    scoop_medium_farleft()
                    log_event("scoop", "medium farleft")
                elif 0.08 <= weight_needed < 0.2:
                    scoop_tiny_right()
                    log_event("scoop", "tiny right")
                elif 0.04 <= weight_needed < 0.08:
                    scoop_tiny_left()
                    log_event("scoop", "tiny left")
                else:
                    log_event("warning",
                              "Remainder too small – manual adjustment")
                    print("⚠️ Remainder too small for automatic scooping")
                    break

                medium_scoop_count += 1
                weight_dispensed = wait_for_stable_weight(previous_weight)
                weight_needed    = calc_weight_needed(weight_dispensed,
                                                      target_weight)
                log_event("scoop", f"medium #{medium_scoop_count}",
                          weight=weight_dispensed,
                          added=weight_dispensed - previous_weight)
                previous_weight = weight_dispensed

            # Final result Case A
            difference = target_weight - weight_dispensed
            if difference < -0.02:
                log_event("result",
                          f"OVERSHOT by {abs(difference):.4f}g",
                          weight=weight_dispensed)
                print(f"⚠️ Overshot! {weight_dispensed:.4f}g "
                      f"(by {abs(difference):.4f}g)")
            elif -0.02 <= difference <= 0.02:
                log_event("result",
                          f"TARGET REACHED, off by {abs(difference):.4f}g",
                          weight=weight_dispensed)
                print(f"✅ Done! {weight_dispensed:.4f}g "
                      f"(off by {abs(difference):.4f}g)")
            else:
                log_event("result",
                          f"UNDERSHOT by {difference:.4f}g",
                          weight=weight_dispensed)
                print(f"❓ Undershot. {weight_dispensed:.4f}g "
                      f"({difference:.4f}g still needed)")
            replace_medium_spatula()

        else:
            log_event("error",
                      f"Fraction {fraction_left:.4f} outside expected range")
            print("Error: fraction outside expected range")

    # ============================================================
    # CASE B: target < 1g → medium spatula only
    # ============================================================
    else:
        medium_scoop_count = 0

        while weight_needed > 0.02:

            if medium_scoop_count >= MAX_MEDIUM_SCOOPS:
                log_event("warning",
                          f"Medium limit reached ({MAX_MEDIUM_SCOOPS})"
                          f" – container probably empty!")
                print("⚠️ Stopping – check container!")
                break

            if weight_needed >= 0.6:
                scoop_medium_left()
                log_event("scoop", "medium left")
            elif 0.4 <= weight_needed < 0.6:
                scoop_medium_middle()
                log_event("scoop", "medium middle")
            elif 0.3 <= weight_needed < 0.4:
                scoop_medium_right()
                log_event("scoop", "medium right")
            elif 0.2 <= weight_needed < 0.3:
                scoop_medium_farleft()
                log_event("scoop", "medium farleft")
            elif 0.08 <= weight_needed < 0.2:
                scoop_tiny()
                log_event("scoop", "tiny middle")
            elif 0.02 < weight_needed < 0.08:
                scoop_fn = random.choice([scoop_tiny_left, scoop_tiny_right])
                log_event("scoop", f"tiny random ({scoop_fn.__name__})")
                scoop_fn()
            else:
                log_event("warning",
                          "Remainder too small – manual adjustment")
                break

            medium_scoop_count += 1
            weight_dispensed = wait_for_stable_weight(previous_weight)
            weight_needed    = calc_weight_needed(weight_dispensed,
                                                  target_weight)
            log_event("scoop", f"medium #{medium_scoop_count}",
                      weight=weight_dispensed,
                      added=weight_dispensed - previous_weight)
            previous_weight = weight_dispensed

        # Final result Case B
        difference = target_weight - weight_dispensed
        if difference < -0.02:
            log_event("result",
                      f"OVERSHOT by {abs(difference):.4f}g",
                      weight=weight_dispensed)
            print(f"⚠️ Overshot! {weight_dispensed:.4f}g "
                  f"(by {abs(difference):.4f}g)")
        elif -0.02 <= difference <= 0.02:
            log_event("result",
                      f"TARGET REACHED, off by {abs(difference):.4f}g",
                      weight=weight_dispensed)
            print(f"✅ Done! {weight_dispensed:.4f}g "
                  f"(off by {abs(difference):.4f}g)")
        else:
            log_event("result",
                      f"UNDERSHOT by {difference:.4f}g",
                      weight=weight_dispensed)
            print(f"❓ Undershot. {weight_dispensed:.4f}g "
                  f"({difference:.4f}g still needed)")
        replace_medium_spatula()

In [ ]:
# Main Workflow

def weighing_workflow(vial_number, target_weight):
    """
    Full workflow:
    1. Pick up vial → place on scale   (Robot_wrapper)
    2. Place funnel over vial          (position file)
    3. Weigh powder                    (weighing logic)
    4. Replace funnel                  (position file)

    Usage:
        run_weighing_workflow("Vial1", 1.5)
    """
    log.clear()
    log_event("info",
              f"Starting | Vial: {vial_number} | Target: {target_weight}g")

    # Step 1 – Pick up vial and place on scale
    log_event("info", f"Picking up {vial_number}")
    robot.PickUpVial(vial_number)

    # Step 2 – Place funnel
    log_event("info", "Placing funnel")
    place_funnel()

    # Step 3 – Zero scale and weigh
    scale.tare()
    log_event("info", "Scale tared – starting weighing")
    run_weighing_logic(target_weight)

    # Step 4 – Replace funnel
    log_event("info", "Replacing funnel")
    replace_funnel()

    # Step 5 – TODO: put vial back
    # robot.PutBackVial(vial_number) ← not yet implemented
    log_event("info", "⚠️ Remember to manually remove vial from scale!")

    print_run_summary()

In [ ]:
# ── Only weighing (vial already in scale) ─────────────────────────
run_weighing_logic(target_weight=1.5)

In [ ]:
# ── Run full experiment ───────────────────────────────────────────
weighing_workflow("Vial1", target_weight=1.5)